# DiffBrush, generatie en evaluatie van Nederlandse adresregels

Parallel aan [`diffusionpen_eval.ipynb`](diffusionpen_eval.ipynb) zodat de twee modellen vergeleken kunnen worden op dezelfde vijf Nederlandse adressen en dezelfde OCR-readback metric.

**Hoe DiffBrush verschilt van DiffusionPen**
- **Regel-gebaseerd, niet woord-gebaseerd.** Eén model call produceert een `64×1024` afbeelding van een volledige tekstregel. Geen woord-voor-woord compositie nodig, want het verbinden van letters over woordgrenzen heen is precies het punt van de paper.
- **Style is altijd image-gebaseerd.** Er is geen 339-writer embedding fallback. Elke generatie heeft één echte handgeschreven regel nodig als style reference. Lokaal hebben we 161 IAM writers in [`DiffBrush/test_data/`](DiffBrush/test_data/).
- **Content is glyph-image gebaseerd.** Tekst wordt gerenderd naar een stack van `16×16` unifont bitmaps en gaat niet door een text tokenizer.
- **De pretrained weights staan NIET in deze repo.** De DiffBrush README linkt naar een Google Drive checkpoint die naar [`DiffBrush/model_zoo/`](DiffBrush/model_zoo/) gedownload moet worden voordat deze notebook iets kan doen na de imports. Cel 2 vertelt je precies waar je het moet plaatsen.

**Wat deze notebook produceert (zodra de checkpoint op zijn plek staat)**
- 15 gegenereerde adresblokken (5 Nederlandse adressen × 3 IAM writers) in `DiffBrush/generated_dutch/`.
- Een visueel grid, een OCR-readback CER/WER tabel en een side-by-side strip tegenover `AddressBlock examples/`.
- Afsluitende markdown cell met een directe vergelijking tegen de DiffusionPen baseline cijfers.

## 1 · Imports, path setup en checkpoint check

In [ ]:
import os, sys, json, random, copy, math
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
import matplotlib.pyplot as plt

# Vind de DiffBrush folder, of de notebook nu geopend is vanuit de project root of vanuit DiffBrush/.
DIFFBRUSH_DIR = Path.cwd()
if (DIFFBRUSH_DIR / 'generate.py').exists() and (DIFFBRUSH_DIR / 'models' / 'unet.py').exists():
    pass
elif (DIFFBRUSH_DIR / 'DiffBrush' / 'generate.py').exists():
    DIFFBRUSH_DIR = DIFFBRUSH_DIR / 'DiffBrush'
else:
    raise FileNotFoundError(
        f'Kon DiffBrush/generate.py niet vinden vanuit {Path.cwd()}. '
        'Open deze notebook vanuit de DiffBrush folder of de project root.'
    )
sys.path.insert(0, str(DIFFBRUSH_DIR))
os.chdir(DIFFBRUSH_DIR)
print('Working dir:', os.getcwd())

from models.unet import UNetModel
from models.diffusion import Diffusion
from data_loader.IAMDataset import IAMGenerateDataset
from diffusers import AutoencoderKL
import torchvision
import cv2

print('torch', torch.__version__, '| CUDA beschikbaar:', torch.cuda.is_available())

# --- Checkpoint check ---
MODEL_ZOO = DIFFBRUSH_DIR / 'model_zoo'
ckpts = sorted([p for p in MODEL_ZOO.glob('*.pt')] + [p for p in MODEL_ZOO.glob('*.pth')])
if not ckpts:
    print('\n' + '=' * 72)
    print('GEEN DIFFBRUSH CHECKPOINT GEVONDEN')
    print('=' * 72)
    print(f'Verwacht ergens onder: {MODEL_ZOO}')
    print('Download vanaf de link in DiffBrush/README.md (Google Drive),')
    print('plaats het .pt bestand in DiffBrush/model_zoo/ en run opnieuw.')
    print('Volgende cellen zullen falen totdat dat bestand bestaat.\n')
    PRETRAINED = None
else:
    PRETRAINED = ckpts[0]
    print(f'Checkpoint gevonden: {PRETRAINED.relative_to(DIFFBRUSH_DIR)}')

## 2 · Config en seeds

In [ ]:
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

# Model dimensies uit configs/IAM.yml.
MODEL_CFG = SimpleNamespace(
    in_channels=4, out_channels=4, emb_dim=512,
    num_res_blocks=1, num_heads=4,
    nb_classes=496,           # WRITER_NUMS uit generate.py
    sampling_timesteps=50,    # DDIM steps
    eta=0.0,                  # DDIM eta
    fixed_len=1024,           # Output breedte
    img_h=64,                 # Output hoogte
)

# Pak een lokale stable-diffusion-v1-5 als die ergens naast staat, anders de HF hub id.
_candidates = [
    DIFFBRUSH_DIR / 'model_zoo',                                    # conventie in DiffBrush README
    DIFFBRUSH_DIR / 'stable-diffusion-v1-5',
    DIFFBRUSH_DIR.parent / 'DiffusionPen' / 'stable-diffusion-v1-5',
]
STABLE_DIF_PATH = next((str(p) for p in _candidates if (p / 'vae').exists()),
                       'stable-diffusion-v1-5/stable-diffusion-v1-5')
print('Device :', device)
print('SD path:', STABLE_DIF_PATH)

## 3 · Laad de VAE

Dezelfde SD-v1.5 VAE die DiffusionPen ook gebruikte. Bij de eerste run wordt deze van HF gedownload als er geen lokale kopie op disk staat.

In [ ]:
vae = AutoencoderKL.from_pretrained(STABLE_DIF_PATH, subfolder='vae').to(device)
vae.requires_grad_(False).eval()
print('VAE klaar.')

## 4 · Bouw de UNet en laad de DiffBrush checkpoint

Zelfde constructor als `generate.py`. `nb_classes` is alleen relevant tijdens training. Bij inference wordt de writer identiteit volledig gecodeerd via de style image.

In [ ]:
unet = UNetModel(
    in_channels=MODEL_CFG.in_channels,
    model_channels=MODEL_CFG.emb_dim,
    out_channels=MODEL_CFG.out_channels,
    num_res_blocks=MODEL_CFG.num_res_blocks,
    attention_resolutions=(1, 1),
    channel_mult=(1, 1),
    num_heads=MODEL_CFG.num_heads,
    context_dim=MODEL_CFG.emb_dim,
    nb_classes=MODEL_CFG.nb_classes,
).to(device)

if PRETRAINED is None:
    raise FileNotFoundError(
        'Geen checkpoint in DiffBrush/model_zoo/. Zie het bericht in cel 1. Niets hierna zal werken.'
    )
state = torch.load(PRETRAINED, map_location='cpu')
# Sommige checkpoints zijn gewrapped in DDP en hebben een 'module.' prefix. Strip die als die er is.
if any(k.startswith('module.') for k in state.keys()):
    state = {k[len('module.'):]: v for k, v in state.items()}
missing, unexpected = unet.load_state_dict(state, strict=False)
unet.eval()
print(f'Geladen {PRETRAINED.name} | missing={len(missing)} unexpected={len(unexpected)}')
if missing:    print('  eerste missing :', missing[:3])
if unexpected: print('  eerste unexpect:', unexpected[:3])

## 5 · Diffusion sampler en dataset helpers

We instantieren `IAMGenerateDataset` puur voor de `get_content(text)` en de unifont bitmaps. Style references komen van een kleine inline helper in plaats van de iteration logica van de dataset (die geeft alle writers tegelijk terug).

In [ ]:
diffusion = Diffusion(device=device)

# Instantieer de generate-dataset alleen voor de unifont-glyph helpers.
# `ref_num` doet er niet toe, want we itereren er niet doorheen.
_ds = IAMGenerateDataset(style_path=str(DIFFBRUSH_DIR / 'test_data'), type='test', ref_num=1)

AVAILABLE_WRITERS = sorted(p.name for p in (DIFFBRUSH_DIR / 'test_data').iterdir() if p.is_dir())
print(f'{len(AVAILABLE_WRITERS)} writers beschikbaar, bijvoorbeeld {AVAILABLE_WRITERS[:6]}')

# DiffBrush is getraind op wikitext regels van lengte 35-61 (zie generate.py),
# EN op korte regels die getiled werden door `concat_short_img` (base_dataset.py)
# zodat ZOWEL de content transcript als de image 2 tot 5 keer herhaald werden
# om het 1024 px canvas te vullen. Naïef korte content zoals "Voorhofstraat 12"
# (16 karakters) voeden raakt hetzelfde regime, maar in plaats van schone herhalingen
# krijgen we verminkte echo's ("GACKOWKKK", "DEN DEN", "der der der") omdat:
#   - de cross-attention weinig content tokens over te veel pixels verspreidt, en
#   - PAD/space padding het model onzeker laat over waar leeg "moet" eindigen.
#
# De fix die hier gebruikt wordt: spiegel expliciet de training augmentatie.
# Tile de tekst met enkele spaties als scheiding tot de totale lengte in distributie zit
# (>= TARGET_TILE_LEN), voer de getilede content, en crop dan de eerste tile.
# De verwachte breedte van de eerste tile is `1024 * text_length / tiled_length`
# (het model verspreidt getilede content uniform), dus de crop is precies in plaats
# van heuristisch.
TARGET_TILE_LEN = 45  # middelpunt van de 35-61 training distributie

def load_style_ref(writer_id: str, prefer_wide: bool = True) -> torch.Tensor:
    """Laad één style image als `(1, 1, 64, fixed_len)` tensor, genormaliseerd naar [0, 1].

    Volgt wat `IAMGenerateDataset.get_style_ref` doet: padt tot fixed_len met wit.
    Kiest standaard een bredere sample (>512 px breed), want zo is het model getraind
    en korte refs verliezen context.
    """
    wdir = DIFFBRUSH_DIR / 'test_data' / writer_id
    paths = list(wdir.glob('*.png'))
    if not paths:
        raise FileNotFoundError(f'Geen style refs in {wdir}')
    if prefer_wide:
        wide = [p for p in paths if cv2.imread(str(p), cv2.IMREAD_GRAYSCALE).shape[1] > 512]
        if wide:
            paths = wide
    p = random.choice(paths)
    arr = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0
    h, w = arr.shape
    canvas = np.ones((MODEL_CFG.img_h, MODEL_CFG.fixed_len), dtype=np.float32)
    canvas[:h, :min(w, MODEL_CFG.fixed_len)] = arr[:, :MODEL_CFG.fixed_len]
    return torch.from_numpy(canvas).unsqueeze(0).unsqueeze(0)  # (1, 1, 64, 1024)

def _tile_text(text: str, target_len: int = TARGET_TILE_LEN) -> tuple[str, int]:
    """Herhaal `text` met enkele spaties als scheiding tot de totale lengte >= target_len is.
    Geeft (tiled_text, n_tiles) terug. Voor tekst die al lang genoeg is geeft het (text, 1) terug.
    """
    if len(text) >= target_len:
        return text, 1
    n_tiles = max(1, math.ceil((target_len + 1) / (len(text) + 1)))
    tiled = (text + ' ') * (n_tiles - 1) + text
    return tiled, n_tiles

def text_to_content(text: str) -> tuple[torch.Tensor, int, int]:
    """Geeft (content tensor, n_tiles, tiled_len) terug. content is (1, T, 16, 16)."""
    tiled, n_tiles = _tile_text(text)
    for ch in tiled:
        if ch not in _ds.letter2index:
            raise ValueError(
                f'Karakter {ch!r} zit niet in het DiffBrush IAM alfabet. '
                f'Toegestaan: {_ds.letters!r}'
            )
    idx_list = [_ds.letter2index[c] for c in tiled]
    content_ref = _ds.con_symbols[idx_list]
    content_ref = 1.0 - content_ref
    return content_ref.unsqueeze(0), n_tiles, len(tiled)

# Smoke check (raakt het diffusion model niet, alleen de helpers).
_s = load_style_ref(AVAILABLE_WRITERS[0])
_c, _n, _tl = text_to_content('Den Haag')
print('style shape  :', tuple(_s.shape))
print(f'content shape: {tuple(_c.shape)} (n_tiles={_n}, tiled_len={_tl}, first_tile_px~{int(MODEL_CFG.fixed_len * 8 / _tl)})')

## 6 · Genereer één volledige regel

`generate_line` draait de DDIM sampler één keer en geeft een strak gecropte grayscale PIL afbeelding van de regel terug. Trailing whitespace (alles rechts van de laatste inkt kolom) wordt verwijderd. De verticale dimensie blijft op de volle 64 px.

In [ ]:
def crop_first_tile(
    img: Image.Image,
    text_length: int,
    n_tiles: int,
    tiled_len: int,
    ink_threshold: int = 200,
    min_island_width: int = 4,
    tile_safety_px: int = 4,
) -> Image.Image:
    """Crop naar de eerste tile op basis van de exacte proportionele breedte.

    De cross-attention van DiffBrush verspreidt getilede content uniform over het
    1024 px canvas, dus `text_length / tiled_len * 1024` is de verwachte
    horizontale breedte van de eerste tile (alleen tekst, niet de trailing
    inter-tile ruimte). We croppen er net iets voorbij (`tile_safety_px`) om
    de trailing edge van het laatste karakter te behouden (bijvoorbeeld de punt
    van een `i` of de staart van een `g`) zonder de eerste letter van de
    volgende tile mee te nemen.

    Wanneer `n_tiles == 1` (tekst zit al in distributie) valt dit terug op een
    ink-bbox crop met een ruime length-based right cap.
    """
    arr = np.array(img.convert('L'))
    col_has_ink = (arr < ink_threshold).any(axis=0)
    if not col_has_ink.any():
        return img

    runs = np.diff(np.concatenate([[0], col_has_ink.view(np.int8), [0]]))
    starts = np.where(runs == 1)[0]
    ends = np.where(runs == -1)[0]
    keep = (ends - starts) >= min_island_width
    starts, ends = starts[keep], ends[keep]
    if len(starts) == 0:
        return img

    left = int(starts[0])
    if n_tiles > 1:
        right_cap = int(MODEL_CFG.fixed_len * text_length / tiled_len) + tile_safety_px
    else:
        right_cap = left + text_length * 33
    right = min(int(ends[-1]), right_cap)
    return img.crop((left, 0, right, img.height))


# Cache van ruwe 1024 px brede regel generaties, geïndexeerd op (text, writer_id).
# Hierdoor kan de crop-en-stack stap in cel 9 afgesteld worden zonder dat de
# diffusion opnieuw moet draaien.
_RAW_LINE_CACHE: dict[tuple[str, str], tuple[Image.Image, int, int]] = {}

@torch.no_grad()
def generate_line(text: str, writer_id: str, crop: bool = True) -> Image.Image:
    """Draai DiffBrush één keer met getilede content en geef een grayscale PIL image van de eerste tile terug."""
    key = (text, writer_id)
    if key in _RAW_LINE_CACHE:
        raw, n_tiles, tiled_len = _RAW_LINE_CACHE[key]
    else:
        style = load_style_ref(writer_id).to(device)                              # (1, 1, 64, 1024)
        content, n_tiles, tiled_len = text_to_content(text)
        content = content.to(device)                                              # (1, T, 16, 16)
        x = torch.randn((1, MODEL_CFG.in_channels,
                         MODEL_CFG.img_h // 8, MODEL_CFG.fixed_len // 8), device=device)
        sampled = diffusion.ddim_sample(
            unet, vae, n=1, x=x, styles=style, content=content,
            sampling_timesteps=MODEL_CFG.sampling_timesteps, eta=MODEL_CFG.eta,
        )                                                                         # (1, 3, 64, 1024) in [0, 1]
        raw = torchvision.transforms.ToPILImage()(sampled.squeeze(0)).convert('L')
        _RAW_LINE_CACHE[key] = (raw, n_tiles, tiled_len)
    return crop_first_tile(raw, text_length=len(text), n_tiles=n_tiles,
                           tiled_len=tiled_len) if crop else raw

# Smoke test: één regel. Slaat op naar disk voor visuele bevestiging als plt.show niets doet.
_demo = generate_line('Voorhofstraat 12', AVAILABLE_WRITERS[0])
_demo.save(DIFFBRUSH_DIR / '_smoke_diffbrush.png')
plt.figure(figsize=(12, 1.5))
plt.imshow(_demo, cmap='gray'); plt.axis('off')
plt.title(f'smoke test: "Voorhofstraat 12", writer {AVAILABLE_WRITERS[0]}'); plt.show()
print('size:', _demo.size)

## 7 · Stack drie gegenereerde regels tot een adresblok

Zelfde min-blend en jitter aanpak als de DiffusionPen notebook, maar dan alleen op regel niveau. DiffBrush regelt de compositie binnen een regel zelf al voor ons.

In [ ]:
LINE_H = 64
LINE_GAP_MIN, LINE_GAP_MAX = 6, 14
BG = 255

def _blend_paste(canvas: np.ndarray, patch: np.ndarray, x: int, y: int) -> None:
    """Min-blend `patch` op `canvas` op positie (x, y). Voorkomt zichtbare naden tussen patches."""
    ph, pw = patch.shape
    H, W = canvas.shape
    y0, y1 = max(0, y), min(H, y + ph)
    x0, x1 = max(0, x), min(W, x + pw)
    if y0 >= y1 or x0 >= x1:
        return
    py0, py1 = y0 - y, y1 - y
    px0, px1 = x0 - x, x1 - x
    canvas[y0:y1, x0:x1] = np.minimum(canvas[y0:y1, x0:x1], patch[py0:py1, px0:px1])

def render_block(lines: list[str], writer_id: str, seed: int | None = None) -> Image.Image:
    rng = np.random.default_rng(seed if seed is not None else hash(writer_id) & 0xFFFFFFFF)
    line_arrs = [np.array(generate_line(ln, writer_id)) for ln in lines]
    line_gaps = [int(rng.integers(LINE_GAP_MIN, LINE_GAP_MAX + 1)) for _ in range(len(line_arrs) - 1)]
    W = max(a.shape[1] for a in line_arrs)
    H = sum(a.shape[0] for a in line_arrs) + sum(line_gaps)
    canvas = np.full((H, W), BG, dtype=np.uint8)
    y = 0
    for i, a in enumerate(line_arrs):
        _blend_paste(canvas, a, 0, y)
        y += a.shape[0] + (line_gaps[i] if i < len(line_gaps) else 0)
    return Image.fromarray(canvas)

## 8 · Nederlandse test adressen en gekozen writers

In [ ]:
ADDRESSES = [
    ['Karol Gackowski',       'Voorhofstraat 12',         '2511 AB Den Haag'],
    ['Joep Houweling',        'Laan van Meerdervoort 81', '2517 AS Den Haag'],
    ['Jurrien de Knecht',     'Hoflaan 4 B',              '3032 AB Rotterdam'],
    ['Prime Vision BV',       'Olof Palmestraat 14',      '2616 LR Delft'],
    ['Famille van der Berg',  'Kerkstraat 27 III',        '1011 AB Amsterdam'],
]

WRITERS = [AVAILABLE_WRITERS[0], AVAILABLE_WRITERS[len(AVAILABLE_WRITERS) // 2], AVAILABLE_WRITERS[-1]]
print('Writers:', WRITERS)
print(f'{len(ADDRESSES)} adressen x {len(WRITERS)} writers = {len(ADDRESSES) * len(WRITERS)} blokken om te genereren')

## 9 · Genereer alle blokken (trage cel)

In [ ]:
OUT_DIR = DIFFBRUSH_DIR / 'generated_dutch'
OUT_DIR.mkdir(exist_ok=True)

samples = {}
for ai, lines in enumerate(ADDRESSES):
    for w in WRITERS:
        print(f'[adr {ai+1}/{len(ADDRESSES)}] writer {w}: {" / ".join(lines)}')
        img = render_block(lines, w)
        out_path = OUT_DIR / f'addr{ai:02d}_writer{w}.png'
        img.save(out_path)
        samples[(ai, w)] = img
print('Opgeslagen naar', OUT_DIR)

## 10 · Visueel grid

In [ ]:
fig, axes = plt.subplots(len(ADDRESSES), len(WRITERS),
                         figsize=(5 * len(WRITERS), 2.0 * len(ADDRESSES)))
if len(ADDRESSES) == 1:
    axes = np.array([axes])
for ai in range(len(ADDRESSES)):
    for wi, w in enumerate(WRITERS):
        ax = axes[ai, wi]
        ax.imshow(samples[(ai, w)], cmap='gray')
        ax.set_xticks([]); ax.set_yticks([])
        if ai == 0:
            ax.set_title(f'writer {w}')
        if wi == 0:
            ax.set_ylabel(f'adr {ai}', rotation=0, ha='right', va='center')
plt.tight_layout(); plt.show()

## 11 · OCR readback, CER en WER tegenover de bedoelde tekst

Zelfde setup als de DiffusionPen notebook, zodat de cijfers direct vergelijkbaar zijn.

In [ ]:
def levenshtein(a: str, b: str) -> int:
    if a == b: return 0
    if not a:  return len(b)
    if not b:  return len(a)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i] + [0] * len(b)
        for j, cb in enumerate(b, 1):
            cur[j] = min(cur[j-1] + 1, prev[j] + 1, prev[j-1] + (ca != cb))
        prev = cur
    return prev[-1]

def cer(ref: str, hyp: str) -> float:
    return levenshtein(ref, hyp) / max(1, len(ref))

def wer(ref: str, hyp: str) -> float:
    rs, hs = ref.split(), hyp.split()
    if not rs:
        return 1.0 if hs else 0.0
    prev = list(range(len(hs) + 1))
    for i, ra in enumerate(rs, 1):
        cur = [i] + [0] * len(hs)
        for j, hb in enumerate(hs, 1):
            cur[j] = min(cur[j-1] + 1, prev[j] + 1, prev[j-1] + (ra != hb))
        prev = cur
    return prev[-1] / len(rs)

ocr_engine = None
try:
    import easyocr
    _reader = easyocr.Reader(['en', 'nl'], gpu=torch.cuda.is_available())
    def ocr(img: Image.Image) -> str:
        return ' '.join(_reader.readtext(np.array(img.convert('RGB')), detail=0, paragraph=True)).strip()
    ocr_engine = 'easyocr'
except Exception as e1:
    try:
        import pytesseract
        def ocr(img):
            return pytesseract.image_to_string(img, lang='nld+eng').strip()
        ocr_engine = 'pytesseract'
    except Exception as e2:
        print('Geen OCR engine. easyocr of pytesseract is nodig.')
        print(' easyocr error:', e1)
        print(' pytesseract error:', e2)
        ocr = None
print('OCR engine:', ocr_engine)

In [ ]:
try:
    from IPython.display import display
except ImportError:
    display = print

results = []
if ocr is not None:
    for ai, lines in enumerate(ADDRESSES):
        ref = ' '.join(lines)
        for w in WRITERS:
            hyp = ocr(samples[(ai, w)])
            results.append({
                'addr': ai, 'writer': w,
                'reference': ref, 'ocr': hyp,
                'CER': round(cer(ref.lower(), hyp.lower()), 3),
                'WER': round(wer(ref.lower(), hyp.lower()), 3),
            })
    try:
        import pandas as pd
        df = pd.DataFrame(results)
        display(df)
        print('Gemiddelde CER:', round(df['CER'].mean(), 3), '| Gemiddelde WER:', round(df['WER'].mean(), 3))
    except ImportError:
        for r in results:
            print(r)
        print('Gemiddelde CER:', round(np.mean([r['CER'] for r in results]), 3),
              '| Gemiddelde WER:', round(np.mean([r['WER'] for r in results]), 3))
else:
    print('(OCR overgeslagen, geen engine beschikbaar)')

## 12 · Realisme, side-by-side met echte Prime Vision samples

In [ ]:
REAL_DIR = DIFFBRUSH_DIR.parent / 'AddressBlock examples'
real_paths = sorted(REAL_DIR.glob('real-hw-*.png'))[:5]
print(f'Toon {len(real_paths)} echte samples vs {len(ADDRESSES)} gegenereerd (writer {WRITERS[0]})')

rows = max(len(real_paths), len(ADDRESSES))
fig, axes = plt.subplots(rows, 2, figsize=(12, 2.0 * rows))
for i in range(rows):
    if i < len(real_paths):
        axes[i, 0].imshow(Image.open(real_paths[i]).convert('L'), cmap='gray')
    axes[i, 0].set_xticks([]); axes[i, 0].set_yticks([])
    if i == 0: axes[i, 0].set_title('Echte Prime Vision sample')

    if i < len(ADDRESSES):
        axes[i, 1].imshow(samples[(i, WRITERS[0])], cmap='gray')
    axes[i, 1].set_xticks([]); axes[i, 1].set_yticks([])
    if i == 0: axes[i, 1].set_title(f'DiffBrush, writer {WRITERS[0]}')
plt.tight_layout(); plt.show()